# Gaussian Quadrature

Going through the Gaussian quadrature section of Numerical Recipes to get a hang out how Gaussian quadrature methods are done. Particularly so I can adapt them to handle numerical integrals of Gaussians.

In [9]:
import numpy as np
from scipy.special import legendre

## W(x) = 1 and N=10

This is a simple quadrature routine that uses tabulated abscissas and weights for the case of W(x) = 1 and N=10. So integration of all polynomials should be exact for this function.

In [3]:
def qgaus(func, a, b):
    '''Perform Gaussian Quadrature integration for W(x)=1 and 10 points.'''
    ABSCISSA = np.array([
        0.1488743389816312, 0.4333953941292472, 0.6794095682990244, 0.8650633666889845, 0.9739065285171717])
    WEIGHTS = np.array([
        0.2955242247147529, 0.2692667193099963, 0.2190863625159821, 0.1494513491505806, 0.0666713443086881])
    x_midpoint = (a + b) * 0.5
    x_range = (a - b) * 0.5
    x_scaled = np.concatenate((-ABSCISSA[::-1], ABSCISSA)) * x_range + x_midpoint
    f_values = func(x_scaled)
    full_weights = np.concatenate((WEIGHTS[::-1], WEIGHTS))
    return sum(f_values * full_weights) * x_range

In [6]:
qgaus(lambda x: x**2, 0, 2)

-2.666666666666667

In [14]:
def gauss_legendre_weights(a, b, n):
    '''Return the abscissa and weights for Gauss-Legendre quadrature with n points.'''
    x_midpoint = (a + b) / 2
    x_range = (b - a) / 2
    poly_leg = legendre(n)
    roots = np.sort(np.roots(poly_leg))
    poly_leg_der = np.polyder(poly_leg)
    weights = 2 / (1 - roots**2) / (np.polyval(poly_leg_der, roots))**2
    scaled_weights = weights * x_range
    abscissa = roots * x_range + x_midpoint
    return abscissa, scaled_weights
    

In [15]:
gauss_legendre_weights(-1, 1, 10)

(array([-0.97390653, -0.86506337, -0.67940957, -0.43339539, -0.14887434,
         0.14887434,  0.43339539,  0.67940957,  0.86506337,  0.97390653]),
 array([0.06667134, 0.14945135, 0.21908636, 0.26926672, 0.29552422,
        0.29552422, 0.26926672, 0.21908636, 0.14945135, 0.06667134]))

In [17]:
def self_legendre(n):
    '''Calculate the legendre polynomial of order n using the recurrence relation.'''
    P_prev = 0
    P_cur = np.poly1d([1])
    x_poly = np.poly1d([1, 0])
    for j in range(n):
        P_next = ((2*j + 1) * x_poly * P_cur - j * P_prev) / (j+1)
        P_prev = P_cur
        P_cur = P_next
    return P_cur

In [25]:
n=6
print(self_legendre(n))
print(legendre(n))

       6         4         2
14.44 x - 19.69 x + 6.562 x - 0.3125
       6         4             3         2
14.44 x - 19.69 x + 1.603e-15 x + 6.562 x - 0.3125


In [38]:
def normalized_hermite_recurrence(n):
    '''Return the normlaized Hermite polynimial of order n using the recurrence relation.'''
    H_prev = 0
    H_cur = 1 / np.pi**(0.25) * np.poly1d([1])
    x_poly = np.poly1d([1, 0])
    for j in range(n):
        H_next = x_poly * np.sqrt(2/(j+1)) * H_cur - np.sqrt(j/(j+1)) * H_prev
        H_prev = H_cur
        H_cur = H_next
    return H_cur

In [39]:
def gauss_hermite_weights(n):
    '''Return the abscissa and weights for Gauss-Hermite integration by quadrature.'''
    poly_herm = normalized_hermite_recurrence(n)
    roots = np.sort(np.roots(poly_herm))
    poly_herm_der = np.polyder(poly_herm)
    weights = 2 / poly_herm_der(roots)**2
    return roots, weights

In [40]:
def qgaus_legendre(func, n):
    '''Use Gaussian Quadrature to calculate integrals with a Gaussian weighting factor.'''
    ABSCISSA, WEIGHTS = gauss_hermite_weights(n)
    f_values = func(ABSCISSA)
    return sum(f_values * WEIGHTS)

In [64]:
qgaus_legendre(lambda x: x**10/np.sqrt(np.pi), 10)

29.53124999999975

## Quadrature

Now we include the reucrrence relations in a quadrature routine so that we can achieve a requested tolerance.